# PixelForge: Image Restoration & 2x Super-Resolution Exploratory Notebook

This notebook visualizes **True Ground Truth (256x256)** vs **Corrupt Noisy LR Input (128x128 upsampled)** vs **Restored Predictions (256x256)**, along with pixel intensity distribution histograms.

In [ ]:
import os
import sys
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

sys.path.append('..')
from data.dataset import PairedImageDataset
from models.builder import build_model
from utils import calculate_psnr, calculate_ssim_tensor, plot_comparison_grid, plot_intensity_histograms

%matplotlib inline

In [ ]:
# Initialize Validation Dataset
gt_dir = '../data/sample_dataset/val/gt'
noisy_dir = '../data/sample_dataset/val/noisy'

dataset = PairedImageDataset(gt_dir, noisy_dir, is_train=False)
print(f'Loaded dataset with {len(dataset)} samples.')

In [ ]:
# Load Model Architecture (UNet or ESRGAN)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = build_model(model_name='unet', in_channels=3, out_channels=3).to(device)
checkpoint_path = '../checkpoints/best_unet.pth'
if os.path.exists(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    print('Checkpoint loaded successfully.')
else:
    print('Checkpoint not found; using untrained weights for demo.')

model.eval()

In [ ]:
# Visual Inspection & Triplet Grid
sample_idx = 0
noisy_tensor, gt_tensor, filename = dataset[sample_idx]
noisy = noisy_tensor.unsqueeze(0).to(device)
gt = gt_tensor.unsqueeze(0).to(device)

with torch.no_grad():
    restored = model(noisy)

psnr_val = calculate_psnr(restored[0], gt[0])
ssim_val = calculate_ssim_tensor(restored[0], gt[0])

gt_np = gt[0].cpu().numpy().transpose(1, 2, 0)
restored_np = restored[0].cpu().numpy().transpose(1, 2, 0)
corrupt_upsampled = F.interpolate(noisy, size=(256, 256), mode='bilinear', align_corners=False)
corrupt_np = corrupt_upsampled[0].cpu().numpy().transpose(1, 2, 0)

plot_comparison_grid(gt_np, corrupt_np, restored_np, psnr_val=psnr_val, ssim_val=ssim_val, show=True)

In [ ]:
# Pixel Intensity Distribution Histograms
plot_intensity_histograms(gt_np, corrupt_np, restored_np, show=True)